In [1]:
print("\n... IMPORTS STARTING ...\n")

print("\n\tVERSION INFORMATION")
import tensorflow as tf

print(f"\t\t– TENSORFLOW VERSION: {tf.__version__}")

import pandas as pd

pd.options.mode.chained_assignment = None
print(f"\t\t– PANDAS VERSION: {pd.__version__}")

import numpy as np

print(f"\t\t– NUMPY VERSION: {np.__version__}")

import sklearn

print(f"\t\t– SKLEARN VERSION: {sklearn.__version__}")

from sklearn.preprocessing import RobustScaler, PolynomialFeatures
from sklearn.model_selection import GroupKFold, StratifiedKFold

import os
import random
import gc
import re
import sys
import json
import math
import time
import zipfile
import pathlib
import warnings
import hashlib
import requests
import urllib
import pickle
import shutil
import csv
import glob
from glob import glob

from tqdm import tqdm

tqdm.pandas()


def seed_it_all(seed=7):
    """Attempt to make runs reproducible."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)


print("\n\n... IMPORTS COMPLETE ...\n")





... IMPORTS STARTING ...


	VERSION INFORMATION


2026-05-06 12:40:09.623038: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778071209.637399      55 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778071209.641851      55 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-06 12:40:09.661287: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

		– TENSORFLOW VERSION: 2.18.0
		– PANDAS VERSION: 2.2.3
		– NUMPY VERSION: 1.26.4
		– SKLEARN VERSION: 1.2.2


... IMPORTS COMPLETE ...



In [2]:
print("\n... BASIC DATA SETUP STARTING ...\n\n")

DATA_DIR = "/kaggle/input/uw-madison-gi-tract-image-segmentation"
TRAIN_DIR = os.path.join(DATA_DIR, "train")
TRAIN_CSV = os.path.join(DATA_DIR, "train.csv")
train_df = pd.read_csv(TRAIN_CSV)

all_train_images = glob(os.path.join(TRAIN_DIR, "**", "*.png"), recursive=True)

print("\n... ORIGINAL TRAINING DATAFRAME... \n")
display(train_df.head())

TEST_DIR = os.path.join(DATA_DIR, "test")
SS_CSV = os.path.join(DATA_DIR, "sample_submission.csv")
ss_df = pd.read_csv(SS_CSV)

all_test_images = glob(os.path.join(TEST_DIR, "**", "*.png"), recursive=True)

DEBUG = len(ss_df) == 0

if DEBUG:
    TEST_DIR = TRAIN_DIR
    all_test_images = all_train_images
    ss_df = train_df.iloc[:10].copy()
    ss_df = ss_df[["id", "class"]].copy()
    ss_df["predicted"] = ""

print("\n\n\n... ORIGINAL SUBMISSION DATAFRAME... \n")
display(ss_df.head())

SF2LF = {"lb": "Large Bowel", "sb": "Small Bowel", "st": "Stomach"}
LF2SF = {v: k for k, v in SF2LF.items()}
print(f"\n\n\n... ARE WE DEBUGGING: {DEBUG}... \n")

print("\n... BASIC DATA SETUP FINISHED ...\n\n")





... BASIC DATA SETUP STARTING ...



... ORIGINAL TRAINING DATAFRAME... 



,id,class,segmentation
0,case77_day20_slice_0001,large_bowel,NaN
1,case77_day20_slice_0001,small_bowel,NaN
2,case77_day20_slice_0001,stomach,NaN
3,case77_day20_slice_0002,large_bowel,NaN
4,case77_day20_slice_0002,small_bowel,NaN





... ORIGINAL SUBMISSION DATAFRAME... 



,id,class,predicted
0,case123_day20_slice_0001,large_bowel,1 1 5 2
1,case123_day20_slice_0001,small_bowel,1 1 5 2
2,case123_day20_slice_0001,stomach,1 1 5 2
3,case123_day20_slice_0002,large_bowel,1 1 5 2
4,case123_day20_slice_0002,small_bowel,1 1 5 2





... ARE WE DEBUGGING: False... 


... BASIC DATA SETUP FINISHED ...




In [3]:
def get_filepath_from_partial_identifier(_ident, file_list):
    return [x for x in file_list if _ident in x][0]


def df_preprocessing(df, globbed_file_list, is_test=False):
    """Add useful columns and merge file paths / RLE masks."""
    df["case_id_str"] = df["id"].apply(lambda x: x.split("_", 2)[0])
    df["case_id"] = df["id"].apply(
        lambda x: int(x.split("_", 2)[0].replace("case", ""))
    )

    df["day_num_str"] = df["id"].apply(lambda x: x.split("_", 2)[1])
    df["day_num"] = df["id"].apply(lambda x: int(x.split("_", 2)[1].replace("day", "")))

    df["slice_id"] = df["id"].apply(lambda x: x.split("_", 2)[2])

    # Build a partial identifier that matches the folder structure
    df["_partial_ident"] = (
        globbed_file_list[0].rsplit("/", 4)[0]
        + "/"
        + df["case_id_str"]
        + "/"
        + df["case_id_str"]
        + "_"
        + df["day_num_str"]
        + "/scans/"
        + df["slice_id"]
    )

    _tmp_merge_df = pd.DataFrame(
        {
            "_partial_ident": [x.rsplit("_", 4)[0] for x in globbed_file_list],
            "f_path": globbed_file_list,
        }
    )
    df = df.merge(_tmp_merge_df, on="_partial_ident").drop(columns=["_partial_ident"])

    df["slice_h"] = df["f_path"].apply(lambda x: int(x[:-4].rsplit("_", 4)[1]))
    df["slice_w"] = df["f_path"].apply(lambda x: int(x[:-4].rsplit("_", 4)[2]))
    df["px_spacing_h"] = df["f_path"].apply(lambda x: float(x[:-4].rsplit("_", 4)[3]))
    df["px_spacing_w"] = df["f_path"].apply(lambda x: float(x[:-4].rsplit("_", 4)[4]))

    if not is_test:
        l_bowel_df = df[df["class"] == "large_bowel"][["id", "segmentation"]].rename(
            columns={"segmentation": "lb_seg_rle"}
        )
        s_bowel_df = df[df["class"] == "small_bowel"][["id", "segmentation"]].rename(
            columns={"segmentation": "sb_seg_rle"}
        )
        stomach_df = df[df["class"] == "stomach"][["id", "segmentation"]].rename(
            columns={"segmentation": "st_seg_rle"}
        )
        df = df.merge(l_bowel_df, on="id", how="left")
        df = df.merge(s_bowel_df, on="id", how="left")
        df = df.merge(stomach_df, on="id", how="left")
        df = df.drop_duplicates(subset=["id"]).reset_index(drop=True)

        df["lb_seg_flag"] = df["lb_seg_rle"].apply(lambda x: not pd.isna(x))
        df["sb_seg_flag"] = df["sb_seg_rle"].apply(lambda x: not pd.isna(x))
        df["st_seg_flag"] = df["st_seg_rle"].apply(lambda x: not pd.isna(x))
        df["n_segs"] = (
            df["lb_seg_flag"].astype(int)
            + df["sb_seg_flag"].astype(int)
            + df["st_seg_flag"].astype(int)
        )

    new_col_order = [
        "id",
        "f_path",
        "n_segs",
        "lb_seg_rle",
        "lb_seg_flag",
        "sb_seg_rle",
        "sb_seg_flag",
        "st_seg_rle",
        "st_seg_flag",
        "slice_h",
        "slice_w",
        "px_spacing_h",
        "px_spacing_w",
        "case_id_str",
        "case_id",
        "day_num_str",
        "day_num",
        "slice_id",
    ]
    if is_test:
        new_col_order.insert(1, "class")
    new_col_order = [c for c in new_col_order if c in df.columns]
    df = df[new_col_order]
    return df


# Apply preprocessing
train_df = df_preprocessing(train_df, all_train_images, is_test=False)
ss_df = df_preprocessing(ss_df, all_test_images, is_test=True)

print("\nProcessed training dataframe preview:")
display(train_df.head())
print("\nProcessed submission dataframe preview:")
display(ss_df.head())





Processed training dataframe preview:


,id,f_path,n_segs,lb_seg_rle,lb_seg_flag,sb_seg_rle,sb_seg_flag,st_seg_rle,st_seg_flag,slice_h,slice_w,px_spacing_h,px_spacing_w,case_id_str,case_id,day_num_str,day_num,slice_id
0,case77_day20_slice_0001,/kaggle/input/uw-madison-gi-tract-image-segmen...,0,NaN,False,NaN,False,NaN,False,266,266,1.5,1.5,case77,77,day20,20,slice_0001
1,case77_day20_slice_0002,/kaggle/input/uw-madison-gi-tract-image-segmen...,0,NaN,False,NaN,False,NaN,False,266,266,1.5,1.5,case77,77,day20,20,slice_0002
2,case77_day20_slice_0003,/kaggle/input/uw-madison-gi-tract-image-segmen...,0,NaN,False,NaN,False,NaN,False,266,266,1.5,1.5,case77,77,day20,20,slice_0003
3,case77_day20_slice_0004,/kaggle/input/uw-madison-gi-tract-image-segmen...,0,NaN,False,NaN,False,NaN,False,266,266,1.5,1.5,case77,77,day20,20,slice_0004
4,case77_day20_slice_0005,/kaggle/input/uw-madison-gi-tract-image-segmen...,0,NaN,False,NaN,False,NaN,False,266,266,1.5,1.5,case77,77,day20,20,slice_0005



Processed submission dataframe preview:


,id,class,f_path,slice_h,slice_w,px_spacing_h,px_spacing_w,case_id_str,case_id,day_num_str,day_num,slice_id
0,case123_day20_slice_0001,large_bowel,/kaggle/input/uw-madison-gi-tract-image-segmen...,266,266,1.5,1.5,case123,123,day20,20,slice_0001
1,case123_day20_slice_0001,small_bowel,/kaggle/input/uw-madison-gi-tract-image-segmen...,266,266,1.5,1.5,case123,123,day20,20,slice_0001
2,case123_day20_slice_0001,stomach,/kaggle/input/uw-madison-gi-tract-image-segmen...,266,266,1.5,1.5,case123,123,day20,20,slice_0001
3,case123_day20_slice_0002,large_bowel,/kaggle/input/uw-madison-gi-tract-image-segmen...,266,266,1.5,1.5,case123,123,day20,20,slice_0002
4,case123_day20_slice_0002,small_bowel,/kaggle/input/uw-madison-gi-tract-image-segmen...,266,266,1.5,1.5,case123,123,day20,20,slice_0002


In [4]:
# Build a lookup dictionary from fully‑segmented slices (where all three organs are present)
slice_px_map = {}
subset = train_df[train_df.n_segs == 3]
group_cols = ["slice_h", "slice_w", "px_spacing_h", "px_spacing_w", "slice_id"]
for _, row in (
    subset.groupby(group_cols)[["lb_seg_rle", "sb_seg_rle", "st_seg_rle"]]
    .first()
    .reset_index()
    .iterrows()
):
    key_base = f"{row['slice_h']}-{row['slice_w']}-{row['px_spacing_h']}-{row['px_spacing_w']}-{row['slice_id']}"
    slice_px_map[f"{key_base}-large_bowel"] = row["lb_seg_rle"]
    slice_px_map[f"{key_base}-small_bowel"] = row["sb_seg_rle"]
    slice_px_map[f"{key_base}-stomach"] = row["st_seg_rle"]

print(f"Lookup table contains {len(slice_px_map)} entries.")




Lookup table contains 639 entries.


In [5]:
# Build an identifier matching the keys in slice_px_map and map to the RLE strings
ss_df["ident"] = (
    ss_df["slice_h"].astype(str)
    + "-"
    + ss_df["slice_w"].astype(str)
    + "-"
    + ss_df["px_spacing_h"].astype(str)
    + "-"
    + ss_df["px_spacing_w"].astype(str)
    + "-"
    + ss_df["slice_id"].astype(str)
    + "-"
    + ss_df["class"]
)
ss_df["predicted"] = ss_df["ident"].map(slice_px_map)




In [6]:
# Keep only the required columns and write the submission file
ss_df = ss_df[["id", "class", "predicted"]].copy()
submission_path = "submission.csv"
ss_df.to_csv(submission_path, index=False)
print(f"Submission written to {submission_path}")
display(ss_df.head())




Submission written to submission.csv


,id,class,predicted
0,case123_day20_slice_0001,large_bowel,NaN
1,case123_day20_slice_0001,small_bowel,NaN
2,case123_day20_slice_0001,stomach,NaN
3,case123_day20_slice_0002,large_bowel,NaN
4,case123_day20_slice_0002,small_bowel,NaN


In [7]:
# Quick sanity check – show a few predicted RLE strings
print("Sample predicted RLE values:")
display(ss_df["predicted"].head())

Sample predicted RLE values:


0    NaN
1    NaN
2    NaN
3    NaN
4    NaN
Name: predicted, dtype: object